In [1]:
import torch
import torch.nn.functional as F 
from torch import nn
from torch.utils.data import DataLoader

from torchvision.datasets import ImageFolder
from torchvision.models import vit_b_16, ViT_B_16_Weights

from PIL import Image
import numpy as np

In [2]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

In [3]:
weights = ViT_B_16_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

In [4]:
train_ds = ImageFolder(root="intel-image-classification/seg_train", transform=preprocess)
valid_ds = ImageFolder(root="intel-image-classification/seg_test", transform=preprocess)
mini_batch_size = 64
train_dl = DataLoader(train_ds, batch_size=mini_batch_size, shuffle=True, drop_last=False, num_workers=4)
valid_dl = DataLoader(train_ds, batch_size=mini_batch_size, num_workers=4)

In [5]:
model = vit_b_16(weights=weights)
model.heads.head = torch.nn.Linear(model.heads.head.in_features, 6)
model = model.to(device)
model

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [6]:
class WrappedDataLoader:
    def __init__(self, dl, func):
        self.dl = dl
        self.func = func

    def __len__(self):
        return len(self.dl)

    def __iter__(self):
        for b in self.dl:
            yield (self.func(*b))


def put_to_gpu(x, y):
    return x.to(device), y.to(device)

In [7]:
for param in model.parameters():
    param.requires_grad = False
for param in model.heads.head.parameters():
    param.requires_grad = True

In [8]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

4614

In [9]:
optimizer = torch.optim.Adam(model.parameters())

In [10]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

In [11]:
early_stopping = EarlyStopping(patience=3, delta=0.01)

In [12]:
def fit(epochs, model, optimizer, train_dl, valid_dl=None):
    loss_func = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        for X_mb, y_mb in train_dl:
            y_hat = model(X_mb)

            loss = loss_func(y_hat, y_mb)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        model.eval()
        with torch.no_grad():
            train_loss = sum(loss_func(model(X_mb), y_mb) for X_mb, y_mb in train_dl)
            valid_loss = sum(loss_func(model(X_mb), y_mb) for X_mb, y_mb in valid_dl)
        print('epoch {}, training loss {}'.format(epoch + 1, train_loss / len(train_dl)))
        print('epoch {}, validation loss {}'.format(epoch + 1, valid_loss / len(valid_dl)))

        early_stopping(valid_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break

    print('Finished training')

    return model

In [13]:
epochs = 10

model = fit(epochs, model, optimizer, WrappedDataLoader(train_dl, put_to_gpu), WrappedDataLoader(valid_dl, put_to_gpu))

epoch 1, training loss 0.20838989317417145
epoch 1, validation loss 0.2079785317182541
epoch 2, training loss 0.18229949474334717
epoch 2, validation loss 0.18231201171875
epoch 3, training loss 0.16658595204353333
epoch 3, validation loss 0.1665394902229309
epoch 4, training loss 0.1536109447479248
epoch 4, validation loss 0.15401799976825714
epoch 5, training loss 0.14614702761173248
epoch 5, validation loss 0.14638486504554749
epoch 6, training loss 0.1403178721666336
epoch 6, validation loss 0.13999657332897186
epoch 7, training loss 0.13458365201950073
epoch 7, validation loss 0.13479380309581757
epoch 8, training loss 0.13239863514900208
epoch 8, validation loss 0.13224710524082184
epoch 9, training loss 0.13016627728939056
epoch 9, validation loss 0.13009758293628693
epoch 10, training loss 0.1236681267619133
epoch 10, validation loss 0.12389066815376282
Finished training


In [14]:
early_stopping.load_best_model(model)
model = model.cpu()

In [15]:
test_image = Image.open("intel-image-classification/seg_test/sea/21191.jpg")
print("original image's shape: " + str(test_image.size))
transformed_img = preprocess(test_image)
print("transformed image's shape: " + str(transformed_img.shape))
# form a batch with only one image
batch_img = torch.unsqueeze(transformed_img, 0)
print("image batch's shape: " + str(batch_img.shape))

output = model.to('cpu')(batch_img)

print("output vector's shape: " + str(output.shape))
percentage = F.softmax(output, dim=1)[0] * 100.0
_, indices = torch.sort(output, descending=True)

# map the class no. to the corresponding label
with open('intel-image-classification/class_names_Intel.txt') as labels:
    classes = [i.strip() for i in labels.readlines()]
results = [(classes[i], percentage[i].item()) for i in indices[0][:5]]

for i in range(3):
    print('{}: {:.4f}%'.format(results[i][0], results[i][1]))

original image's shape: (150, 150)
transformed image's shape: torch.Size([3, 224, 224])
image batch's shape: torch.Size([1, 3, 224, 224])
output vector's shape: torch.Size([1, 6])
sea: 99.6609%
glacier: 0.3036%
mountain: 0.0238%


In [16]:
def evaluate(model, data_loader):    
    model.eval()
    accuracy = 0
    with torch.no_grad():
        for X, y in data_loader:
            y_hat = model(X).cpu().numpy()
            y_hat = np.argmax(y_hat, axis=1)
            accuracy += (y_hat == y.cpu().numpy()).mean()
    accuracy /= len(data_loader)

    return accuracy

In [17]:
test_ds = ImageFolder(root="intel-image-classification/seg_test", transform=preprocess)
test_dl = DataLoader(test_ds, batch_size=mini_batch_size*2)
accuracy = evaluate(model.to(device), WrappedDataLoader(test_dl, put_to_gpu))
print("accuracy: " + str(accuracy))

accuracy: 0.9373604910714285


AlexNet: 91.87% accuracy